In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


In [2]:
# ============================================================
# GALAXY / STAR / QSO CLASSIFICATION
# XGBoost + LightGBM + Neural Network + Logistic Regression stacking
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import ExtraTreesClassifier

from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ============================================================
# LOAD DATA AND CONFIGURATION
# ============================================================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

TARGET = "class"
SEED = 42
N_FOLDS = 5


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def feature_engineering(df):
    print("Computing Feature Engineering")
    df = df.copy()
    required_cols = ['u','g','r','i','z', 'redshift','alpha','delta'    ]

    if all(col in df.columns for col in required_cols):
        # -----------------------------------
        # Color Indices
        # -----------------------------------
        df["u_g"] = df["u"] - df["g"]
        df["g_r"] = df["g"] - df["r"]
        df["r_i"] = df["r"] - df["i"]
        df["i_z"] = df["i"] - df["z"]

        df["u_r"] = df["u"] - df["r"]
        df["u_i"] = df["u"] - df["i"]
        df["u_z"] = df["u"] - df["z"]

        df["g_i"] = df["g"] - df["i"]
        df["g_z"] = df["g"] - df["z"]

        df["r_z"] = df["r"] - df["z"]

        # -----------------------------------
        # Magnitude Statistics
        # -----------------------------------
        mags = ['u','g','r','i','z']
        df["mag_mean"] = df[mags].mean(axis=1)
        df["mag_std"] = df[mags].std(axis=1)
        df["mag_min"] = df[mags].min(axis=1)
        df["mag_max"] = df[mags].max(axis=1)
        df["mag_range"] = (df["mag_max"] - df["mag_min"]        )

        # -----------------------------------
        # Redshift Features
        # -----------------------------------
        df["redshift_abs"] = np.abs(df["redshift"])
        df["redshift_sq"] = (df["redshift"] ** 2        )
        df["redshift_log"] = np.log1p(np.abs(df["redshift"])        )

        # -----------------------------------
        # Coordinate Features
        # -----------------------------------

        df["alpha_rad"] = np.radians(df["alpha"])
        df["delta_rad"] = np.radians(df["delta"])
        df["sin_alpha"] = np.sin(df["alpha_rad"])
        df["cos_alpha"] = np.cos(df["alpha_rad"])
        df["sin_delta"] = np.sin(df["delta_rad"])
        df["cos_delta"] = np.cos(df["delta_rad"])

        # -----------------------------------
        # Interaction Features
        # -----------------------------------
        df["redshift_gr"] = (df["redshift"] * df["g_r"])
        df["redshift_ri"] = (df["redshift"] * df["r_i"])
        df["redshift_iz"] = (df["redshift"] * df["i_z"])

    return df


# ============================================================
# APPLY FEATURE ENGINEERING
# ============================================================

train = feature_engineering(train)
test = feature_engineering(test)

X = train.drop(columns=[TARGET])
y = train[TARGET]

test_ids = test["id"]

X = X.drop(columns=["id"])
X_test = test.drop(columns=["id"])


# ============================================================
# TARGET ENCODING
# ============================================================

le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)

print("\nClasses:")
print(le.classes_)

# ============================================================
# IDENTIFY FEATURE TYPES
# ============================================================

numeric_features = X.select_dtypes(include=np.number).columns.tolist()

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))

print("\nNumerical Features:", numeric_features)
print("Categorical Features:", categorical_features)

print("\nCategorical Columns:")
print(categorical_features)


# ============================================================
# PREPROCESSING
# ============================================================

numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


Train Shape: (577347, 12)
Test Shape : (247435, 11)
Computing Feature Engineering
Computing Feature Engineering

Classes:
['GALAXY' 'QSO' 'STAR']

Numeric Features: 35
Categorical Features: 2

Numerical Features: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'mag_mean', 'mag_std', 'mag_min', 'mag_max', 'mag_range', 'redshift_abs', 'redshift_sq', 'redshift_log', 'alpha_rad', 'delta_rad', 'sin_alpha', 'cos_alpha', 'sin_delta', 'cos_delta', 'redshift_gr', 'redshift_ri', 'redshift_iz']
Categorical Features: ['spectral_type', 'galaxy_population']

Categorical Columns:
['spectral_type', 'galaxy_population']


In [3]:
"""
xgb = XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)
lgbm = LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced', n_jobs=-1)
cat = CatBoostClassifier(random_seed=42, verbose=0)
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
"""

"\nxgb = XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42)\nlgbm = LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced', n_jobs=-1)\ncat = CatBoostClassifier(random_seed=42, verbose=0)\nrf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)\n"

In [4]:
# ============================================================
# MODELS
# ============================================================
xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED
)

lgbm = LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=127,
    subsample=0.85,
    colsample_bytree=0.85,
    force_col_wise=True,
    verbosity=0,
    random_state=SEED
)

cat = CatBoostClassifier(
    iterations=1200,
    depth=8,
    learning_rate=0.03,
    loss_function="MultiClass",
    verbose=0,
    random_seed=SEED
)

et = ExtraTreesClassifier(
    n_estimators=1000,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1
)

mlp = MLPClassifier(
    hidden_layer_sizes=(512,256),
    learning_rate_init=0.001,
    max_iter=50, 
    random_state=SEED,
    early_stopping=True
)

In [5]:
# ============================================================
# OOF Stacking
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cat_cols = X.select_dtypes(include=['object','category']).columns

from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

models = {
    "cat": cat,
    "xgb": xgb,
    "lgbm": lgbm,
    "et": et,
    "mlp": mlp
}

oof_preds = []
test_preds = []

for name, model in models.items():
    print(f"\nTraining {name}")
    oof = np.zeros((len(X), n_classes))
    preds = np.zeros((len(X_test), n_classes))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_enc)):
        X_tr = X.iloc[tr_idx]
        y_tr = y_enc[tr_idx]
        X_val = X.iloc[val_idx]
        model.fit(X_tr, y_tr)
        oof[val_idx] = (model.predict_proba(X_val))
        preds += (model.predict_proba(X_test) / N_FOLDS)
    oof_preds.append(oof)
    test_preds.append(preds)
    score = accuracy_score(y_enc, np.argmax(oof, axis=1))
    print(f"{name} OOF = {score:.5f}")


Training cat
cat OOF = 0.96572

Training xgb
xgb OOF = 0.96875

Training lgbm
lgbm OOF = 0.96883

Training et
et OOF = 0.96113

Training mlp
mlp OOF = 0.95733


In [6]:
# ============================================================
# STACKING META MODEL
# ============================================================

meta_X = np.hstack(oof_preds + [X.values])
meta_test = np.hstack(test_preds + [X_test.values])

# Meta Learner
meta_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="MultiClass",
    verbose=0
)

meta_model.fit(meta_X, y_enc)
stack_probs = (meta_model.predict_proba(meta_test))

In [7]:
# Blend Stack + Base Models
base_blend = (
      0.19 * test_preds[0]   # CatBoost
    + 0.21 * test_preds[1]   # XGB
    + 0.22 * test_preds[2]   # LGBM
    + 0.20 * test_preds[3]   # ET
    + 0.18 * test_preds[4]   # MLP
)

final_probs = (
      0.75 * stack_probs
    + 0.25 * base_blend
)

In [8]:
# ============================================================
# SUBMISSION
# ============================================================

pred_idx = np.argmax(final_probs, axis=1)

pred_labels = le.inverse_transform(pred_idx)

submission = pd.DataFrame({"id": test_ids, "class": pred_labels})

submission.to_csv("submission.csv", index=False)

print("\nSubmission Saved")
print(submission.head(10))


Submission Saved
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
5  577352  GALAXY
6  577353  GALAXY
7  577354    STAR
8  577355  GALAXY
9  577356  GALAXY
